In [0]:
# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
CONTAINER  = os.getenv("CONTAINER_NAME")
BRONZE_BASE     = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/fraud_bronze"

# Delta paths
IDENTITY_DELTA_PATH     = f"{BRONZE_BASE}/delta/identity/"
TRANSACTIONS_DELTA_PATH = f"{BRONZE_BASE}/delta/transactions/"

# Metastore database
DATABASE = "fraud_lakehouse"

# ---------------------------------------------------------------------------
# 1. SPARK OPTIMIZATIONS (no storage key — Unity Catalog handles auth)
# ---------------------------------------------------------------------------
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",   "true")
spark.conf.set("spark.sql.adaptive.enabled",                   "true")

print("✅  Spark optimizations configured.")

# ---------------------------------------------------------------------------
# 2. CREATE DATABASE (idempotent)
# ---------------------------------------------------------------------------
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
spark.sql(f"USE {DATABASE}")

print(f"✅  Database '{DATABASE}' ready.")

# ---------------------------------------------------------------------------
# 3. READ RAW PARQUET FILES
# ---------------------------------------------------------------------------
identity_raw     = spark.read.parquet(f"{BRONZE_BASE}/identity/")
transactions_raw = spark.read.parquet(f"{BRONZE_BASE}/transactions/")

print(f"✅  Identity rows      : {identity_raw.count():,}")
print(f"✅  Transaction rows   : {transactions_raw.count():,}")

# ---------------------------------------------------------------------------
# 4. SCHEMA INSPECTION
# ---------------------------------------------------------------------------
print("\n── Identity schema ──────────────────────────────────")
identity_raw.printSchema()

print("\n── Transactions schema ──────────────────────────────")
transactions_raw.printSchema()

# ---------------------------------------------------------------------------
# 5. BASIC QUALITY GATE
# ---------------------------------------------------------------------------
assert identity_raw.count() > 0,     "ERROR: Identity parquet is empty!"
assert transactions_raw.count() > 0, "ERROR: Transactions parquet is empty!"

# ---------------------------------------------------------------------------
# 6. WRITE TO DELTA LAKE — BRONZE TABLES
# ---------------------------------------------------------------------------
(
    identity_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", IDENTITY_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.bronze_identity")
)

(
    transactions_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", TRANSACTIONS_DELTA_PATH)
    .saveAsTable(f"{DATABASE}.bronze_transactions")
)

print(f"\n✅  Delta Bronze tables written to : {BRONZE_BASE}/delta/")
print(f"    ├── bronze_identity     : {spark.table(f'{DATABASE}.bronze_identity').count():,} rows")
print(f"    └── bronze_transactions : {spark.table(f'{DATABASE}.bronze_transactions').count():,} rows")
print(f"\n✅  Metastore tables registered under database '{DATABASE}'")
print("▶   Run notebook 02_silver_transform.py next.")

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-7684759011943558>, line 4
      1 # ---------------------------------------------------------------------------
      2 # 0. CONFIGURATION — using Databricks Secrets
      3 # ---------------------------------------------------------------------------
----> 4 STORAGE_ACCOUNT = dbutils.secrets.get(scope="fraud-scope", key="storage-account")
      5 CONTAINER       = dbutils.secrets.get(scope="fraud-scope", key="container-name")
      6 BRONZE_BASE     = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/fraud_bronze"

File /databricks/python_shell/lib/dbruntime/dbutils.py:258, in DBUtils.SecretsHandler.get(self, scope, key)
    257 def get(self, scope, key):
--> 258     return self.entry_point.getDbutils().preview().secret().get(scope, key)

File /databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway